# Phase 3 Preparation - Daily CMIP6 Export for Selected Models

This notebook exports **daily** NASA/GDDP-CMIP6 data for the Phase 2 selected models so the later QDM bias correction and ETCCDI extreme-index analysis can be performed correctly.

The previous `cmip6_SA.ipynb` produced annual CMIP6 grids. Annual grids are useful for model screening, but daily ETCCDI indices require daily `pr`, `tasmax`, and `tasmin`.

## Cell 1 - Required Packages and Earth Engine Setup

In [ ]:
from pathlib import Path
import gc
import traceback

import ee
import geemap
import geopandas as gpd
import pandas as pd
import xarray as xr
import xee

try:
    ee.Initialize()
except Exception:
    ee.Authenticate()
    ee.Initialize()

print('Required packages loaded: ee, geemap, geopandas, pandas, xarray, xee')

## Cell 2 - AOI and Output Folders

In [ ]:
ROOT = (Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve())
states_shp = ROOT / 'shp' / 'SAsiaFinalD.shp'

output_root = ROOT / 'output' / 'cmip6_daily'
manifest_dir = output_root / '_manifest'
output_root.mkdir(parents=True, exist_ok=True)
manifest_dir.mkdir(parents=True, exist_ok=True)

if not states_shp.exists():
    raise FileNotFoundError(f'Missing shapefile: {states_shp}')

gdf = gpd.read_file(states_shp, encoding='latin1').to_crs('EPSG:4326')
roi = geemap.geopandas_to_ee(gdf)
roi_geom = roi.geometry()

print(f'AOI features: {len(gdf)}')
print(f'AOI bounds: {gdf.total_bounds}')
print(f'Output root: {output_root}')

## Cell 3 - Export Plan

Full three-variable models are selected from Phase 2. `CESM2` is kept as optional precipitation-only because the local archive and NASA/GDDP-CMIP6 availability indicate missing `tasmax` and `tasmin` for this model.

In [ ]:
DATASET_ID = 'NASA/GDDP-CMIP6'
EXPORT_SCALE_M = 25000

FULL_MODELS = [
    'CanESM5',
    'GFDL-ESM4',
    'INM-CM5-0',
    'IPSL-CM6A-LR',
    'MPI-ESM1-2-HR',
]

PRECIP_ONLY_MODELS = ['CESM2']

VARIABLES_BY_MODEL = {model: ['pr', 'tasmax', 'tasmin'] for model in FULL_MODELS}
for model in PRECIP_ONLY_MODELS:
    VARIABLES_BY_MODEL[model] = ['pr']

SCENARIOS = {
    'historical': (1985, 2014),
    'ssp245': (2015, 2100),
    'ssp585': (2015, 2100),
}

# Yearly files are slower to create but much safer than one huge 2015-2100 file.
YEARS_PER_FILE = 1

print('Models and variables:')
for model, variables in VARIABLES_BY_MODEL.items():
    print(f'  {model}: {variables}')

## Cell 4 - Helper Functions

In [ ]:
def variable_metadata(variable):
    meta = {
        'pr': ('Daily precipitation flux converted to precipitation depth', 'mm day-1'),
        'tasmax': ('Daily maximum near-surface air temperature', 'degC'),
        'tasmin': ('Daily minimum near-surface air temperature', 'degC'),
    }
    return meta[variable]


def scenario_year_chunks(start_year, end_year, years_per_file=1):
    chunks = []
    y = start_year
    while y <= end_year:
        y2 = min(y + years_per_file - 1, end_year)
        chunks.append((y, y2))
        y = y2 + 1
    return chunks


def convert_daily_image(image, variable):
    image = ee.Image(image)
    if variable == 'pr':
        # NASA/GDDP-CMIP6 pr is kg m-2 s-1. Convert to mm day-1.
        out = image.multiply(86400.0).rename(variable)
    else:
        # tasmax/tasmin are Kelvin. Convert to degrees C for ETCCDI tools.
        out = image.subtract(273.15).rename(variable)
    return ee.Image(out.copyProperties(image, image.propertyNames()))


def build_daily_collection(model, scenario, variable, start_year, end_year):
    start = f'{start_year}-01-01'
    end = f'{end_year + 1}-01-01'
    col = (
        ee.ImageCollection(DATASET_ID)
        .filter(ee.Filter.eq('model', model))
        .filter(ee.Filter.eq('scenario', scenario))
        .filterDate(start, end)
        .filterBounds(roi_geom)
        .select(variable)
        .map(lambda img: convert_daily_image(ee.Image(img), variable).clip(roi_geom))
    )
    return col


def expected_days(start_year, end_year):
    return len(pd.date_range(f'{start_year}-01-01', f'{end_year}-12-31', freq='D'))


def output_path(model, scenario, variable, start_year, end_year):
    model_dir = output_root / scenario / model / variable
    model_dir.mkdir(parents=True, exist_ok=True)
    suffix = f'{start_year}' if start_year == end_year else f'{start_year}_{end_year}'
    return model_dir / f'{model}_{variable}_{scenario}_{suffix}_daily.nc'


def append_manifest(row):
    manifest_path = manifest_dir / 'cmip6_daily_export_manifest.csv'
    df = pd.DataFrame([row])
    if manifest_path.exists():
        df.to_csv(manifest_path, mode='a', header=False, index=False)
    else:
        df.to_csv(manifest_path, index=False)

## Cell 5 - Define Xee Grid from Existing Annual CMIP6 Template

In [ ]:
template_file = ROOT / 'output' / 'cmip6' / 'historical' / 'CanESM5' / 'CanESM5_pr_historical_1985_2014.nc'

with xr.open_dataset(template_file) as template_ds:
    template_lon = template_ds['lon'].values
    template_lat = template_ds['lat'].values

dx = float(template_lon[1] - template_lon[0])
dy = float(template_lat[1] - template_lat[0])
x_origin = float(template_lon[0] - dx / 2)
y_origin = float(template_lat[0] - dy / 2)

XEE_CRS = 'EPSG:4326'
XEE_CRS_TRANSFORM = (dx, 0, x_origin, 0, dy, y_origin)
XEE_SHAPE_2D = (len(template_lon), len(template_lat))  # xee expects (width, height)

print('Xee grid derived from annual CMIP6 template:')
print('  crs:', XEE_CRS)
print('  crs_transform:', XEE_CRS_TRANSFORM)
print('  shape_2d:', XEE_SHAPE_2D)
print('  lon range:', float(template_lon.min()), float(template_lon.max()))
print('  lat range:', float(template_lat.min()), float(template_lat.max()))

## Cell 6 - Optional Quick Preview

In [ ]:
preview = ee.Image(
    build_daily_collection('CanESM5', 'historical', 'tasmax', 2000, 2000).first()
).clip(roi_geom)

vis = {'min': -10, 'max': 45, 'palette': ['blue', 'white', 'yellow', 'red']}
preview_map = geemap.Map(basemap='SATELLITE')
preview_map.centerObject(roi_geom, 4)
preview_map.addLayer(preview, vis, 'CanESM5 tasmax daily sample, degC')
preview_map.addLayer(roi.style(color='white', fillColor='00000000', width=2), {}, 'AOI')
preview_map

## Cell 7 - Daily Export Loop

This cell is resumable. Existing non-empty files are skipped. If a failed export leaves an empty/corrupt file, delete that file and rerun the cell.

In [ ]:
saved = []
skipped = []
failed = []

ENCODING = {var: {'zlib': True, 'complevel': 4} for var in ['pr', 'tasmax', 'tasmin']}

for scenario, (scenario_start, scenario_end) in SCENARIOS.items():
    chunks = scenario_year_chunks(scenario_start, scenario_end, YEARS_PER_FILE)
    for model, variables in VARIABLES_BY_MODEL.items():
        for variable in variables:
            for chunk_start, chunk_end in chunks:
                out_path = output_path(model, scenario, variable, chunk_start, chunk_end)

                if out_path.exists() and out_path.stat().st_size > 0:
                    print(f'[SKIP] {out_path.relative_to(ROOT)}')
                    skipped.append(str(out_path))
                    continue

                print(f'[PROC] {model} | {scenario} | {variable} | {chunk_start}-{chunk_end}')
                try:
                    daily_col = build_daily_collection(model, scenario, variable, chunk_start, chunk_end)
                    n_images = int(daily_col.size().getInfo())
                    n_expected = expected_days(chunk_start, chunk_end)

                    if n_images == 0:
                        raise RuntimeError('Earth Engine returned zero images')
                    if n_images != n_expected:
                        print(f'  [WARN] image count {n_images}, expected {n_expected}')

                    ds = xr.open_dataset(
                        daily_col,
                        engine='ee',
                        crs=XEE_CRS,
                        crs_transform=XEE_CRS_TRANSFORM,
                        shape_2d=XEE_SHAPE_2D,
                        n_images=n_images,
                    ).sortby('time')

                    if variable not in ds.data_vars:
                        first_var = list(ds.data_vars)[0]
                        ds = ds.rename({first_var: variable})

                    long_name, units = variable_metadata(variable)
                    ds[variable].attrs.update({
                        'long_name': long_name,
                        'units': units,
                        'source_dataset': DATASET_ID,
                        'model': model,
                        'scenario': scenario,
                        'note': 'Daily export for ETCCDI/QDM workflow; clipped to South Asia AOI.',
                    })

                    ds.to_netcdf(out_path, encoding={variable: ENCODING[variable]})
                    ds.close()
                    gc.collect()

                    row = {
                        'status': 'saved',
                        'model': model,
                        'scenario': scenario,
                        'variable': variable,
                        'start_year': chunk_start,
                        'end_year': chunk_end,
                        'n_images': n_images,
                        'expected_days': n_expected,
                        'path': str(out_path.relative_to(ROOT)),
                        'size_mb': round(out_path.stat().st_size / 1024 / 1024, 2),
                    }
                    append_manifest(row)
                    saved.append(str(out_path))
                    print(f'  [OK] {out_path.name} ({row["size_mb"]} MB)')

                except Exception as exc:
                    failed_msg = f'{model}|{scenario}|{variable}|{chunk_start}-{chunk_end}|{type(exc).__name__}: {exc}'
                    failed.append(failed_msg)
                    append_manifest({
                        'status': 'failed',
                        'model': model,
                        'scenario': scenario,
                        'variable': variable,
                        'start_year': chunk_start,
                        'end_year': chunk_end,
                        'n_images': None,
                        'expected_days': expected_days(chunk_start, chunk_end),
                        'path': str(out_path.relative_to(ROOT)),
                        'size_mb': None,
                        'error': failed_msg,
                    })
                    print(f'  [FAIL] {failed_msg}')
                    traceback.print_exc(limit=2)

print(f'Saved: {len(saved)}')
print(f'Skipped: {len(skipped)}')
print(f'Failed: {len(failed)}')

## Cell 8 - Export Inventory

In [ ]:
files = sorted(output_root.rglob('*.nc'))
total_mb = 0
rows = []

for nc in files:
    size_mb = nc.stat().st_size / 1024 / 1024
    total_mb += size_mb
    rel = nc.relative_to(output_root)
    rows.append({'file': str(rel), 'size_mb': round(size_mb, 2)})

inventory = pd.DataFrame(rows)
inventory.to_csv(manifest_dir / 'cmip6_daily_file_inventory.csv', index=False)

print(f'Files: {len(files)}')
print(f'Total size: {total_mb:.1f} MB')
inventory.tail(20)

## Cell 9 - Sanity Check One Daily File

In [ ]:
sample_files = sorted(output_root.rglob('*.nc'))

if sample_files:
    sample_file = sample_files[0]
    with xr.open_dataset(sample_file) as ds:
        print(sample_file.relative_to(ROOT))
        print(ds)
        for var in ds.data_vars:
            print(var, ds[var].attrs)
else:
    print('No daily NetCDF files found yet.')